# NASA C-MAPSS Turbofan Degradation - Exploratory Data Analysis

Phân tích chuỗi dữ liệu cảm biến đa chiều từ động cơ máy bay mô phỏng C-MAPSS (FD001).
Mục tiêu:
1. Khám phá số lượng động cơ và chu kỳ sống đến khi hỏng (run-to-failure cycle).
2. Nhận diện các sensor suy thoái (degradation trend) và sensor hằng số (zero variance).
3. Trực quan hóa đường cong RUL và phân phối thời gian sống còn lại.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.preprocessing.loader import load_data
from src.preprocessing.label_rul import add_rul_target
from src.preprocessing.clean import prune_low_variance_sensors, clean_dataframe

# 1. Load train dataset
df_train = load_data(subset="FD001", mode="train")
print("Shape:", df_train.shape)
print("Number of Engines:", df_train["engine_id"].nunique())
df_train.head()

In [ ]:
# 2. Calculate RUL and summary statistics
df_train = add_rul_target(df_train, clip_limit=125.0)
max_cycles = df_train.groupby("engine_id")["cycle"].max()
print("Cycle to failure statistics:")
print(max_cycles.describe())

# 3. Plot max cycles distribution
plt.figure(figsize=(10, 4))
plt.hist(max_cycles, bins=25, color="skyblue", edgecolor="black")
plt.title("Distribution of Engine Maximum Lifespans (FD001)")
plt.xlabel("Failure Cycle")
plt.ylabel("Engine Count")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
# 4. Sensor variance analysis (identify constant sensors)
sensor_cols = [c for c in df_train.columns if c.startswith("sensor_")]
var_series = df_train[sensor_cols].var().sort_values()
print("Sensors sorted by variance:")
print(var_series)

# Prune invariant sensors
df_clean, dropped = prune_low_variance_sensors(df_train)
print("Dropped invariant sensors:", dropped)

In [ ]:
# 5. Plot Degradation curves for Engine #1, #2, #3
sample_engines = [1, 2, 3]
plt.figure(figsize=(14, 6))

for eng in sample_engines:
    eng_data = df_clean[df_clean["engine_id"] == eng]
    plt.plot(eng_data["cycle"], eng_data["sensor_2"], label=f"Engine {eng} (T24 Temp)")

plt.title("LPC Outlet Temperature (Sensor 2) Degradation Over Time")
plt.xlabel("Operating Cycle")
plt.ylabel("Sensor 2 Value (°R)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()